# Role sentence adjustments — percentage increase

Calculates the percentage increase in the starting-point sentence attributable to each
defendant role, using the manually reviewed role-adjustment data in
`Role Sentence Adjustments_updated 2026.07.30.xlsx`.

For each non-excluded charge the percentage increase is:

```
pct = (sentence after role - starting point) / starting point
```

Results are reported separately for charges **without** and **with** "Cross-border
trafficking" under Additional Circumstances, so both the plain role adjustment and the
role-plus-cross-border adjustment can be read off.

Inclusion rules (matching the review sheet):

- **Actual trafficker** — all charges noted as "Actual trafficker" under Column M
  ("Defendant's Role").
- **Manager / Organiser** — all charges noted as "Manager / Organiser" under Column M,
  but excluding those with "Cross-border trafficking" under Column N
  ("Additional Circumstances").
- **Operator / Financial Controller** — all charges noted as "Operator / Financial
  Controller" under Column M, but excluding those with "Cross-border trafficking" under
  Column N.

Charges flagged as excluded (`Exclusion = 1` / role "(Excluded)") are dropped.


In [1]:

import numpy as np
import pandas as pd
from pathlib import Path

from linear_interpolation_model import get_notebook_dir

notebook_dir = get_notebook_dir()
path = notebook_dir / "Role Sentence Adjustments_updated 2026.07.30.xlsx"
raw = pd.read_excel(path)

normalized_roles = {
	"Actual trafficker": "Actual trafficker",
	"Manager/organiser": "Manager / Organiser",
	"Operator/financial controller": "Operator / Financial Controller",
}

charges = raw.copy()
charges["role"] = charges["Defendant's Role"].str.strip().map(normalized_roles)
charges["cross_border"] = charges["Additional Circumstances"].eq("Cross-border trafficking")

excluded = charges["Exclusion"].eq(1) | charges["role"].isna()
excluded_charges = charges[excluded]
included_charges = charges[~excluded].copy()

usable = included_charges.dropna(
	subset=["starting_point_total_months", "sentence_after_role_total_months"]
).copy()
usable["pct_increase"] = (
	(usable["sentence_after_role_total_months"] - usable["starting_point_total_months"])
	/ usable["starting_point_total_months"]
	* 100
)

print(f"Total charges: {len(raw)}")
print(f"Excluded charges: {len(excluded_charges)}")
print(f"Included charges: {len(included_charges)} ({len(usable)} with a calculable increase)")


Total charges: 274
Excluded charges: 160
Included charges: 114 (107 with a calculable increase)


In [2]:

def role_summary(frame):
	rows = []
	for (role, cross_border), group in frame.groupby(["role", "cross_border"], sort=False):
		pooled = (
			(group.sentence_after_role_total_months.sum() - group.starting_point_total_months.sum())
			/ group.starting_point_total_months.sum()
			* 100
		)
		rows.append({
			"role": role,
			"cross_border": "yes" if cross_border else "no",
			"charges": len(group),
			"mean_pct": group.pct_increase.mean(),
			"median_pct": group.pct_increase.median(),
			"min_pct": group.pct_increase.min(),
			"max_pct": group.pct_increase.max(),
			"pooled_pct": pooled,
		})
	return pd.DataFrame(rows).set_index(["role", "cross_border"])

summary = role_summary(usable)
from IPython.display import display
display(summary.round(2))


charges  mean_pct  median_pct  \
role                            cross_border                                  
Actual trafficker               no                 86      6.05        5.16   
                                yes                 1     28.57       28.57   
Manager / Organiser             no                 15      6.68        6.25   
                                yes                 2      8.16        8.16   
Operator / Financial Controller yes                 2     10.25       10.25   
                                no                  1      7.79        7.79   

                                              min_pct  max_pct  pooled_pct  
role                            cross_border                                
Actual trafficker               no               0.00    17.14        4.89  
                                yes             28.57    28.57       28.57  
Manager / Organiser             no               1.19    16.67        5.16  
                                yes              7.81     8.51        8.11  
Operator / Financial Controller yes              7.69    12.80       10.48  
                                no               7.79     7.79        7.79

In [3]:

# Per-role overall figures and per-charge detail.
overall = role_summary(usable).groupby("role", sort=False).agg(
	charges=("charges", "sum"),
	mean_pct=("mean_pct", "mean"),
	median_pct=("median_pct", "mean"),
)
print("Overall per role (both cross-border groups combined):")
print(overall.round(2))
print()
print("Per-charge detail (first 15):")
detail_cols = ["neutral_citation", "trial_index", "role", "cross_border",
               "starting_point_total_months", "sentence_after_role_total_months", "pct_increase"]
print(usable[detail_cols].head(15).to_string(index=False))


Overall per role (both cross-border groups combined):
                                 charges  mean_pct  median_pct
role                                                          
Actual trafficker                     87     17.31       16.87
Manager / Organiser                   17      7.42        7.21
Operator / Financial Controller        3      9.02        9.02

Per-charge detail (first 15):
 neutral_citation  trial_index              role  cross_border  starting_point_total_months  sentence_after_role_total_months  pct_increase
[2021] HKCFI 1726            0 Actual trafficker         False                         62.0                              65.0      4.838710
[2021] HKCFI 1726            1 Actual trafficker         False                         94.0                              97.0      3.191489
[2021] HKCFI 3331            0 Actual trafficker         False                        144.0                             156.0      8.333333
[2021] HKCFI 3792            0 Actual tr

In [4]:

# Write an Excel report next to the source data.
report_path = notebook_dir / "role_sentence_adjustments_analysis.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
	summary.reset_index().to_excel(writer, sheet_name="summary", index=False)
	usable[detail_cols].to_excel(writer, sheet_name="charges", index=False)
	excluded_charges[[
		"neutral_citation", "trial_index", "Charge_no", "Defendant_id",
		"Defendant's Role", "Exclusion",
	]].to_excel(writer, sheet_name="excluded charges", index=False)
print("Wrote", report_path)


Wrote /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/role_sentence_adjustments_analysis.xlsx


## Reading the results

- `mean_pct` / `median_pct` — the per-charge percentage increase; the median is the
  robust central estimate for a role.
- `pooled_pct` — the percentage increase of the summed sentences, which is dominated by
  large cases.
- Small groups (e.g. one or two charges) should be read as illustrative only.

The plain (non-cross-border) medians can be compared against the current model defaults:
Actual trafficker 5%, Manager / Organiser 10%, Operator / Financial Controller 15%.
